In [3]:
from pathlib import Path
from safeguard_llm.run_safe_llm import run_safe_llm
from safeguard_llm.utils.save_results import save_results_as_json
from transformers import AutoModelForCausalLM, AutoTokenizer
from safeguard_llm.safety_harness import SafeLLM
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pathlib import Path
from datasets import load_dataset,concatenate_datasets
from safeguard_llm.utils.output_judge import Benchmark_Eval
from safeguard_llm.evaluator import SafetyEvaluator
from dotenv import load_dotenv
import asyncio



In [9]:
dataset = load_dataset("allenai/wildjailbreak", "eval", delimiter="\t", keep_default_na=False)
print(dataset)
ds_true = dataset["train"].filter(lambda elm: elm["label"] == 1)
ds_false = dataset["train"].filter(lambda elm: elm["label"] == 0)
ds_true = ds_true.select(range(10))
ds_false = ds_false.select(range(10))
dataset = concatenate_datasets([ds_true, ds_false])

dataloader = DataLoader(dataset, batch_size=16)
outputs = []
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-1.7B", padding_side="left"
)

safe_model = SafeLLM(
    model, tokenizer, max_gen_len=256, config_path="src/safeguard_llm/config/safe_llm_config.yaml"
)



DatasetDict({
    train: Dataset({
        features: ['adversarial', 'label', 'data_type'],
        num_rows: 2210
    })
})


Loading weights: 100%|██████████| 311/311 [00:06<00:00, 47.72it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2544.27it/s]


In [ ]:
all_labels = []
for batch in tqdm(dataloader): 
    inputs = batch["adversarial"]
    labels = batch["label"]
    all_labels.extend(labels)
    output = safe_model.generate(inputs)
    outputs.extend(output)
print(all_labels)
print(len(all_labels))
for i, res in enumerate(outputs): 
    val = all_labels[i]
    val = val.item()
    if val == 0: 
        res.output_label_gold = False
    res.prompt_label_gold = bool(val)


100%|██████████| 2/2 [00:42<00:00, 21.34s/it]


IndexError: list index out of range

In [8]:

evaluator = SafetyEvaluator(outputs, truth_rule=lambda x,y: x)
metrics = evaluator.get_all_classification_reports()
evaluator.print_all_classification_reports(metrics)